# AI-Powered Student Placement Prediction and Career Guidance System
### End-to-End Google Colab Notebook

This notebook covers: dataset generation, preprocessing, EDA, model training & comparison,
hyperparameter tuning, placement probability prediction, a rule-based skill-gap engine, and
AI career guidance via the **Gemini API**.

Run cells top to bottom. The trained model + scaler are saved to `placement_model.pkl` at the end,
ready for use in the Streamlit app.

## 1. Setup

In [ ]:
!pip install -q xgboost google-generativeai scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

sns.set_style("whitegrid")
np.random.seed(42)


## 2. Dataset Generation

We generate a realistic synthetic dataset of 1200 student records (>1000 required).

**Feature descriptions:**
- `CGPA` — Cumulative GPA (4-10 scale)
- `Tenth_Percentage`, `Twelfth_Percentage` — Board exam percentages
- `Internships` — Number of internships completed (0-4)
- `Projects` — Number of academic/personal projects (0-10)
- `Coding_Skills`, `Communication_Skills`, `Aptitude_Score`, `Technical_Skills` — Assessment scores (0-100)
- `Backlogs` — Pending/failed subjects (0-5)
- `Certifications` — Relevant certifications completed (0-8)
- `Hackathon_Participation` — Hackathons participated in (0-5)
- `Placement_Status` — **Target**: 1 = Placed, 0 = Not Placed

The target is generated from a weighted logistic function of the features plus noise, so the
relationships are realistic but not perfectly deterministic — just like real placement data.

In [ ]:
N = 1200

def generate_dataset(n=N):
    cgpa = np.clip(np.random.normal(7.2, 1.0, n), 4.0, 10.0)
    tenth = np.clip(np.random.normal(78, 10, n), 40, 100)
    twelfth = np.clip(np.random.normal(75, 10, n), 40, 100)
    internships = np.random.poisson(1.1, n).clip(0, 4)
    projects = np.random.poisson(2.5, n).clip(0, 10)
    coding = np.clip(np.random.normal(60, 18, n), 0, 100)
    communication = np.clip(np.random.normal(65, 15, n), 0, 100)
    aptitude = np.clip(np.random.normal(60, 17, n), 0, 100)
    technical = np.clip(np.random.normal(60, 18, n), 0, 100)
    backlogs = np.random.choice([0,1,2,3,4,5], n, p=[0.55,0.2,0.12,0.07,0.04,0.02])
    certifications = np.random.poisson(1.5, n).clip(0, 8)
    hackathons = np.random.poisson(0.8, n).clip(0, 5)

    z = (0.55*(cgpa-7) + 0.015*(tenth-75) + 0.015*(twelfth-75) + 0.35*internships
         + 0.18*projects + 0.025*(coding-60) + 0.02*(communication-60)
         + 0.02*(aptitude-60) + 0.02*(technical-60) - 0.6*backlogs
         + 0.15*certifications + 0.12*hackathons + np.random.normal(0, 1.0, n))
    prob = 1 / (1 + np.exp(-z))
    placement = (prob > 0.5).astype(int)

    df = pd.DataFrame({
        "CGPA": cgpa.round(2), "Tenth_Percentage": tenth.round(2), "Twelfth_Percentage": twelfth.round(2),
        "Internships": internships, "Projects": projects, "Coding_Skills": coding.round(1),
        "Communication_Skills": communication.round(1), "Aptitude_Score": aptitude.round(1),
        "Technical_Skills": technical.round(1), "Backlogs": backlogs, "Certifications": certifications,
        "Hackathon_Participation": hackathons, "Placement_Status": placement,
    })
    for col in ["CGPA", "Coding_Skills", "Communication_Skills", "Aptitude_Score"]:
        idx = df.sample(frac=0.02, random_state=1).index
        df.loc[idx, col] = np.nan
    return df

df = generate_dataset()
df.to_csv("placement_dataset.csv", index=False)
print(f"Generated {len(df)} records")
df.head()


In [ ]:
print(df["Placement_Status"].value_counts(normalize=True))
df.describe()


## 3. Data Preprocessing\n\n1. **Missing values** — imputed with column median (robust to outliers/skew).\n2. **Outliers** — capped using the IQR method (clip, don't drop, to preserve sample size).\n3. **Feature scaling** — `StandardScaler` so distance/gradient-based models (SVM, Logistic Regression) aren't biased by feature scale.\n4. No categorical encoding is needed — all features are numeric.

In [ ]:
FEATURES = ["CGPA","Tenth_Percentage","Twelfth_Percentage","Internships","Projects",
            "Coding_Skills","Communication_Skills","Aptitude_Score","Technical_Skills",
            "Backlogs","Certifications","Hackathon_Participation"]
TARGET = "Placement_Status"

print("Missing values before:\n", df[FEATURES].isna().sum())

df_clean = df.copy()
for col in FEATURES:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in FEATURES:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    df_clean[col] = df_clean[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)

print("\nMissing values after:", df_clean[FEATURES].isna().sum().sum())
df_clean.head()


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12,9))
sns.heatmap(df_clean[FEATURES+[TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
# Insight: CGPA, Internships, Projects, Coding/Technical scores correlate positively with
# placement; Backlogs correlates negatively — matches real-world intuition.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18,12))
for ax, col in zip(axes.flatten(), FEATURES):
    sns.histplot(df_clean[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
plt.tight_layout()
plt.show()
# Insight: most scores are roughly normal; Backlogs/Internships/Certifications are right-skewed counts.


In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x=TARGET, data=df_clean, palette="Set2")
plt.title("Placement Distribution")
plt.xticks([0,1], ["Not Placed","Placed"])
plt.show()
print(df_clean[TARGET].value_counts())
# Insight: dataset is moderately imbalanced toward "Placed", consistent with real placement drives.


In [ ]:
sample_cols = ["CGPA","Coding_Skills","Communication_Skills","Internships","Projects",TARGET]
sns.pairplot(df_clean[sample_cols], hue=TARGET, palette="husl", diag_kind="kde")
plt.show()
# Insight: Placed students cluster toward higher CGPA, coding scores, and project counts.


## 5. Train/Test Split & Scaling

In [ ]:
X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)


## 6. Model Training & Comparison\n\nTraining Logistic Regression, Decision Tree, Random Forest, XGBoost, and SVM, then comparing\nAccuracy, Precision, Recall, F1, ROC-AUC, and Confusion Matrix for each.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "SVM": SVC(probability=True, random_state=42),
}

results = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:,1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }
    fitted_models[name] = model

    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.show()

results_df = pd.DataFrame(results).T.sort_values("F1 Score", ascending=False)
results_df


In [ ]:
best_model_name = results_df.index[0]
print("Best performing model (by F1 Score):", best_model_name)
results_df.style.highlight_max(axis=0, color="lightgreen")


In [ ]:
# Feature importance (tree-based model)
importance_model = fitted_models.get("Random Forest")
importances = pd.Series(importance_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.show()
importances


## 7. Hyperparameter Tuning (GridSearchCV)\n\nTuning the best-performing model family to squeeze out extra performance.

In [ ]:
param_grids = {
    "Random Forest": (RandomForestClassifier(random_state=42), {
        "n_estimators": [100, 200, 300], "max_depth": [None, 5, 10, 15], "min_samples_split": [2,5,10]}),
    "XGBoost": (XGBClassifier(random_state=42, eval_metric="logloss"), {
        "n_estimators": [100,200,300], "max_depth": [3,5,7], "learning_rate": [0.01,0.1,0.2]}),
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), {"C":[0.01,0.1,1,10,100]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42), {"max_depth":[3,5,10,None], "min_samples_split":[2,5,10]}),
    "SVM": (SVC(probability=True, random_state=42), {"C":[0.1,1,10], "kernel":["linear","rbf"]}),
}

base_est, grid = param_grids[best_model_name]
search = GridSearchCV(base_est, grid, cv=5, scoring="f1", n_jobs=-1)
search.fit(X_train_scaled, y_train)

print("Best Params:", search.best_params_)
print("Best CV F1:", search.best_score_)

tuned_model = search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_scaled)
y_prob_tuned = tuned_model.predict_proba(X_test_scaled)[:,1]

before_after = pd.DataFrame({
    "Before Tuning": results[best_model_name],
    "After Tuning": {
        "Accuracy": accuracy_score(y_test, y_pred_tuned),
        "Precision": precision_score(y_test, y_pred_tuned),
        "Recall": recall_score(y_test, y_pred_tuned),
        "F1 Score": f1_score(y_test, y_pred_tuned),
        "ROC-AUC": roc_auc_score(y_test, y_prob_tuned),
    }
})
before_after
# Tuning typically yields modest gains (1-4%) by finding the optimal bias/variance
# tradeoff for this model family on this dataset.


## 8. Save Final Model Artifacts

In [ ]:
final_model = tuned_model  # use the tuned model as production model

with open("placement_model.pkl", "wb") as f:
    pickle.dump({"model": final_model, "scaler": scaler, "features": FEATURES}, f)

print("Saved placement_model.pkl")


## 9. Placement Probability Prediction

In [ ]:
def predict_placement(model, scaler, student: dict):
    x = pd.DataFrame([student])[FEATURES]
    x_scaled = scaler.transform(x)
    prob = model.predict_proba(x_scaled)[0,1]
    label = "Likely to be Placed" if prob >= 0.5 else "Unlikely to be Placed (Needs Improvement)"
    confidence = "High" if (prob >= 0.8 or prob <= 0.2) else ("Moderate" if (prob >= 0.65 or prob <= 0.35) else "Low")
    return label, round(prob*100,1), confidence

sample_student = {
    "CGPA": 8.4, "Tenth_Percentage": 88, "Twelfth_Percentage": 85, "Internships": 2,
    "Projects": 4, "Coding_Skills": 82, "Communication_Skills": 75, "Aptitude_Score": 78,
    "Technical_Skills": 80, "Backlogs": 0, "Certifications": 3, "Hackathon_Participation": 2,
}
label, prob_pct, conf = predict_placement(final_model, scaler, sample_student)
print(f"Placement Probability: {prob_pct}%")
print(f"Prediction: {label}")
print(f"Confidence: {conf}")


## 10. Skill Gap Analysis Engine (Rule-Based)

In [ ]:
RULES = [
    {"feature":"Coding_Skills","threshold":60,"below":True,"area":"Coding / DSA",
     "message":"Your coding score is below par. Practice DSA daily on LeetCode/GeeksforGeeks, aiming for 150+ solved problems."},
    {"feature":"Communication_Skills","threshold":60,"below":True,"area":"Communication",
     "message":"Join group discussions, practice mock interviews, and consider a communication training course."},
    {"feature":"Aptitude_Score","threshold":55,"below":True,"area":"Aptitude",
     "message":"Dedicate time to quantitative & logical reasoning practice (e.g. IndiaBix, RS Aggarwal)."},
    {"feature":"Projects","threshold":2,"below":True,"area":"Portfolio",
     "message":"Build 2-3 solid portfolio projects and host them on GitHub with good READMEs."},
    {"feature":"Internships","threshold":1,"below":True,"area":"Industry Experience",
     "message":"Apply for internships via LinkedIn, Internshala, or your campus placement cell."},
    {"feature":"Technical_Skills","threshold":60,"below":True,"area":"Technical Depth",
     "message":"Strengthen CS fundamentals: OS, DBMS, Networks, OOP — heavily tested in interviews."},
    {"feature":"Certifications","threshold":1,"below":True,"area":"Certifications",
     "message":"Earn relevant certifications (AWS, GCP, Coursera/NPTEL) to strengthen your resume."},
    {"feature":"Backlogs","threshold":0,"below":False,"area":"Academics",
     "message":"Clear pending backlogs as a priority — many recruiters filter these out."},
    {"feature":"CGPA","threshold":6.5,"below":True,"area":"Academics",
     "message":"Your CGPA is below typical recruiter cutoffs (6.5-7.0). Focus on improving grades."},
    {"feature":"Hackathon_Participation","threshold":1,"below":True,"area":"Practical Exposure",
     "message":"Participate in hackathons for hands-on, collaborative project experience."},
]

def analyze_skill_gap(student: dict):
    weaknesses, strengths = [], []
    for r in RULES:
        val = student.get(r["feature"])
        if val is None: continue
        weak = (val < r["threshold"]) if r["below"] else (val > r["threshold"])
        (weaknesses if weak else strengths).append(r if weak else r["area"])
    weaknesses = [{"area": r["area"], "message": r["message"]} for r in weaknesses]
    if not weaknesses:
        weaknesses.append({"area":"General","message":"No major weaknesses detected — keep up consistency and start interview prep early."})
    return {"weaknesses": weaknesses, "strengths": sorted(set(strengths))}

gap_report = analyze_skill_gap(sample_student)
for w in gap_report["weaknesses"]:
    print(f"[{w[\'area\']}] {w[\'message\']}")
print("\nStrengths:", gap_report["strengths"])


## 11. AI Career Guidance Module (Gemini API)

This cell calls Google's Gemini API to generate personalized career guidance:
strengths/weaknesses analysis, technologies to learn, project suggestions,
certifications, interview strategy, and recommended career paths.

**You'll need a Gemini API key** — get one free at https://aistudio.google.com/app/apikey
and paste it below (or set it as a Colab secret named `GEMINI_API_KEY`).

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Option A: Colab secret (recommended) -> Colab sidebar -> key icon -> add GEMINI_API_KEY
# Option B: paste directly (not recommended for shared notebooks)
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

def get_career_guidance(student: dict, placement_label: str, placement_prob: float, gap_report: dict):
    weaknesses_text = "\n".join(f"- {w[\'area\']}: {w[\'message\']}" for w in gap_report["weaknesses"])
    strengths_text = ", ".join(gap_report["strengths"]) or "None clearly identified"

    prompt = f"""
You are an expert career counselor for engineering students. Based on the student profile below,
provide structured career guidance.

Student Profile:
{student}

Placement Prediction: {placement_label} ({placement_prob}% probability)
Identified Strengths: {strengths_text}
Identified Weaknesses:
{weaknesses_text}

Provide your response in clearly labeled sections:
1. Strengths Analysis
2. Weaknesses Analysis
3. Technologies to Learn (list 4-6)
4. Project Ideas (list 3-4, specific and relevant to their profile)
5. Recommended Certifications (list 3-4)
6. Interview Preparation Strategy (a short actionable plan)
7. Suggested Career Paths (rank top 3 from: Software Engineer, Data Analyst, Data Scientist,
   Cloud Engineer, DevOps Engineer, Full Stack Developer, AI/ML Engineer)

Keep it concise, practical, and encouraging.
"""
    response = gemini_model.generate_content(prompt)
    return response.text

career_guidance = get_career_guidance(sample_student, label, prob_pct, gap_report)
print(career_guidance)


## 12. Next Steps

- Use `placement_model.pkl` (model + scaler) inside the Streamlit app (`app/streamlit_app.py`).
- The skill-gap rules and Gemini prompt logic live in `utils/recommendations.py` and can be reused as-is.
- For Streamlit Cloud / GitHub deployment steps, see `README.md` in the project root.